# Grant and proof

Issue a selector-bound grant, sign one holder request, then verify and refuse
changed request context.

Install Deno and register its TypeScript kernel with `deno jupyter --install`.
Open this file in a Jupyter front end such as JupyterLab or VS Code, select
**Deno**, then Run All in a fresh kernel. The cells import exact published
signer **2.0.0** and verifier **2.0.0**.

Fixed times describe demonstration credentials. Generated keys, signatures and
nonces stay in memory. Signing and verification do not grant operational
authority; the host owns trust, live revocation, replay and policy.

## Load the published packages

Check both release identities and the backend Deno selects.


In [ ]:
import assert from "node:assert/strict";
import { Buffer } from "node:buffer";
import { generateKeyPairSync, sign } from "node:crypto";
import * as signer from "npm:@bounded-authority-protocol/signer@2.0.0";
import * as verifier from "npm:@bounded-authority-protocol/verifier@2.0.0";
import signerMetadata from "npm:@bounded-authority-protocol/signer@2.0.0/package.json" with {
  type: "json",
};
import verifierMetadata from "npm:@bounded-authority-protocol/verifier@2.0.0/package.json" with {
  type: "json",
};
assert.equal(signerMetadata.version, "2.0.0");
assert.equal(verifierMetadata.version, "2.0.0");
assert.equal(verifier.BACKEND_ID, "portable");
console.log(
  JSON.stringify({
    signer: signerMetadata.version,
    verifier: verifierMetadata.version,
    backend: verifier.BACKEND_ID,
  }),
);

{"signer":"2.0.0","verifier":"2.0.0","backend":"portable"}


## Prepare real demonstration key handles

Each constructor holds a fresh private key in its closure and exposes only the
key-handle contract. No key material is displayed. Production custody supplies
this same contract.

In [ ]:
function createEd25519Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): signer.KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ed25519");
  const publicBytes = new Uint8Array(
    Buffer.from(publicKey.export({ format: "jwk" }).x!, "base64url"),
  );
  return {
    sign: (signingInput) =>
      new Uint8Array(sign(null, signingInput, privateKey)),
    publicKey: () => publicBytes,
    thumbprint: () =>
      verifier.thumbprint(verifier.jwkFromPublicKey(publicBytes)),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
function createEs256Handle(
  keyId: string,
  role: signer.KeyRole = "holder",
): signer.Es256KeyHandle {
  const { publicKey, privateKey } = generateKeyPairSync("ec", {
    namedCurve: "P-256",
  });
  const { x, y } = publicKey.export({ format: "jwk" }) as {
    x: string;
    y: string;
  };
  const publicBytes = new Uint8Array([
    0x04,
    ...Buffer.from(x, "base64url"),
    ...Buffer.from(y, "base64url"),
  ]);
  return {
    sign: (signingInput) =>
      new Uint8Array(
        sign("sha256", signingInput, {
          key: privateKey,
          dsaEncoding: "ieee-p1363",
        }),
      ),
    publicKey: () => publicBytes,
    thumbprint: () => verifier.v3.thumbprint({ crv: "P-256", kty: "EC", x, y }),
    keyIdentity: () => ({ keyId, publicKey: publicBytes }),
    signingIdentity: () => ({ role, keyId, publicKey: publicBytes }),
  };
}
const ed25519Width =
  (await createEd25519Handle("example-ed25519").publicKey()).length;
const es256Width =
  (await createEs256Handle("example-es256").publicKey()).length;
assert.equal(ed25519Width, 32);
assert.equal(es256Width, 65);
console.log(
  JSON.stringify({ ed25519Bytes: ed25519Width, es256Bytes: es256Width }),
);

{"ed25519Bytes":32,"es256Bytes":65}


## Issue a bounded grant

Only the issuer-role key can issue a grant. This grant names one repository
selector.

In [ ]:
const issuer = createEd25519Handle("issuer-key", "issuer");
const holder = createEd25519Handle("holder-key");
const grant = await signer.signGrant({
  issuer: "https://issuer.example",
  grantId: "urn:example:grant:1",
  audiences: ["https://resource.example"],
  issuedAt: 1_000,
  notBefore: 1_000,
  expiresAt: 2_000,
  holderThumbprint: await holder.thumbprint(),
  operations: [{
    name: "read_file",
    selectors: [{
      kind: "equals",
      path: ["repository"],
      value: { t: "string", v: verifier.strUtf8("example/website") },
    }],
  }],
}, issuer);
assert.ok(grant.ok);
console.log(JSON.stringify({ ok: grant.ok }));

{"ok":true}


## Sign one request

The proof commits to the method, URI, invocation and tagged arguments.


In [ ]:
const request = {
  method: "GET",
  targetUri: "https://resource.example/files",
  invocationId: "0f8d5a5e-7c3b-4e7a-9a51-2f4c1d6b8e90",
  operation: "read_file",
  castArguments: {
    t: "object",
    v: new Map<string, verifier.Tagged>([
      ["repository", { t: "string", v: verifier.strUtf8("example/website") }],
      ["path", { t: "string", v: verifier.strUtf8("README.md") }],
    ]),
  } satisfies verifier.Tagged,
};
const envelope = await signer.signReport(
  { ...request, grantCompact: grant.value.grant },
  holder,
  { issuedAt: 1_100, proofId: "proof-1" },
);
assert.ok(envelope.ok);
console.log(JSON.stringify({ ok: envelope.ok }));

{"ok":true}


## Verify the received context

The resource supplies its already-trusted key and the actual request. A
successful result still leaves authorization unevaluated.

In [ ]:
const expected = {
  ...request,
  trustedIssuer: { keyId: "issuer-key", publicKey: await issuer.publicKey() },
  issuer: "https://issuer.example",
  audience: "https://resource.example",
  evaluationTime: 1_120,
  clockSkew: 0,
  proofMaxAge: 300,
  nonce: { kind: "not_required" } as const,
};
const facts = verifier.checkEnvelope(
  envelope.value.grant,
  envelope.value.proof,
  expected,
);
assert.ok(facts.ok);
assert.equal(facts.value.operation, "read_file");
console.log(
  JSON.stringify({
    ok: facts.ok,
    operation: facts.value.operation,
    authorization: facts.value.authorization,
  }),
);

{"ok":true,"operation":"read_file","authorization":"not_evaluated"}


## Refuse a changed method

Changing the expected method invalidates the original request proof.


In [ ]:
const changed = verifier.checkEnvelope(
  envelope.value.grant,
  envelope.value.proof,
  { ...expected, method: "PUT" },
);
assert.deepEqual(changed, { ok: false });
console.log(JSON.stringify(changed));

{"ok":false}


## Finish

Shut down the kernel to discard the demonstration keys. No private material was
persisted. Continue with the
[getting started guide](../guides/getting-started.md).